In [1]:
# Set this to your checkout of the repo before running the notebook.
REPO_ROOT = "/home/ratnayn/codex/mIF-pipeline/"


In [2]:
from pathlib import Path
import json
import sys
import traceback

if "REPO_ROOT" not in globals():
    raise RuntimeError(
        "Set REPO_ROOT to the repository path before running this cell. Example: REPO_ROOT = '/abs/path/to/mIF-pipeline'"
    )

REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()
if not (REPO_ROOT / "src").exists():
    raise RuntimeError(
        f"REPO_ROOT does not look like the repo root: {REPO_ROOT}. Expected to find {REPO_ROOT / 'src'}."
    )

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repo root: {REPO_ROOT}")
print(f"Source dir added to sys.path: {SRC_DIR}")


Repo root: /home/ratnayn/codex/mIF-pipeline
Source dir added to sys.path: /home/ratnayn/codex/mIF-pipeline/src


In [3]:
from mif_pipeline import crop_channel_images


In [4]:
result = crop_channel_images(
    "/mnt/c/analysis/Data_prototype/SLIDE-0329",
    "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048",
    x=9512,
    y=10157,
    width=2048,
    height=2048,
    patterns=["*.ome.tif"],
    force=True,
)

In [5]:
from tifffile import TiffFile

path = "/mnt/c/analysis/Data_prototype/SLIDE-0329/SLIDE-0329_1.0.4_R000_Cy3_Bit2-RS0584-Cy3B_FINAL_AFR_F.ome.tif"
with TiffFile(path) as tf:
    for s_idx, s in enumerate(tf.series):
        print(f"Series {s_idx}: axes={s.axes}, shape={s.shape!r}")
        # Most WSIs use levels (pyramids) within a series
        levels = getattr(s, "levels", [s])
        for l_idx, lvl in enumerate(levels):
            page = lvl.pages[0]
            tw = getattr(page, "tilewidth", None)
            tl = getattr(page, "tilelength", None)
            rps = getattr(page, "rowsperstrip", None)
            comp = page.compression.name if page.compression else "none"
            kind = "tiled" if (tw and tl) else "striped"
            ts = f"{tw}x{tl}" if (tw and tl) else f"rows/strip={rps}"
            print(f"  Level {l_idx}: {kind}, tile={ts}, size={lvl.shape[-2]}x{lvl.shape[-1]}, compression={comp}")
    print(tf.pages[0].description)

Series 0: axes=YX, shape=(62617, 66406)
  Level 0: tiled, tile=512x512, size=62617x66406, compression=LZW
  Level 1: tiled, tile=512x512, size=31308x33203, compression=LZW
  Level 2: tiled, tile=512x512, size=15654x16601, compression=LZW
  Level 3: tiled, tile=512x512, size=7827x8300, compression=LZW
  Level 4: tiled, tile=512x512, size=3913x4150, compression=LZW
  Level 5: tiled, tile=512x512, size=1956x2075, compression=LZW
  Level 6: tiled, tile=512x512, size=978x1037, compression=LZW
  Level 7: tiled, tile=512x512, size=489x518, compression=LZW
<?xml version="1.0" encoding="utf-8"?>
<OME xmlns:schemaLocation="http://www.openmicroscopy.org/Schemas/OME/2016-06 http://www.openmicroscopy.org/Schemas/OME/2016-06/ome.xsd" xmlns="http://www.openmicroscopy.org/Schemas/OME/2016-06">
  <Image ID="Image:0">
    <Pixels ID="0" Type="uint16" SizeX="66406" SizeY="62617" SizeZ="1" SizeC="1" SizeT="1" PhysicalSizeX="0.325002437518281" PysicalSizeXUnit="um" PhysicalSizeY="0.325002437518281" Dimensi

In [15]:
from tifffile import TiffFile

path = "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/outputs/SLIDE-0329_full_merge.ome.tif"
with TiffFile(path) as tf:
    for s_idx, s in enumerate(tf.series):
        print(f"Series {s_idx}: axes={s.axes}, shape={s.shape!r}")
        # Most WSIs use levels (pyramids) within a series
        levels = getattr(s, "levels", [s])
        for l_idx, lvl in enumerate(levels):
            page = lvl.pages[0]
            tw = getattr(page, "tilewidth", None)
            tl = getattr(page, "tilelength", None)
            rps = getattr(page, "rowsperstrip", None)
            comp = page.compression.name if page.compression else "none"
            kind = "tiled" if (tw and tl) else "striped"
            ts = f"{tw}x{tl}" if (tw and tl) else f"rows/strip={rps}"
            print(f"  Level {l_idx}: {kind}, tile={ts}, size={lvl.shape[-2]}x{lvl.shape[-1]}, compression={comp}")
    print(tf.pages[0].description)


Series 0: axes=CYX, shape=(24, 2048, 2048)
  Level 0: tiled, tile=256x256, size=2048x2048, compression=ADOBE_DEFLATE
  Level 1: tiled, tile=256x256, size=1025x1024, compression=ADOBE_DEFLATE
  Level 2: tiled, tile=256x256, size=513x513, compression=ADOBE_DEFLATE
  Level 3: tiled, tile=256x256, size=257x257, compression=ADOBE_DEFLATE
  Level 4: tiled, tile=256x256, size=129x129, compression=ADOBE_DEFLATE
  Level 5: tiled, tile=256x256, size=65x65, compression=ADOBE_DEFLATE
  Level 6: tiled, tile=256x256, size=33x33, compression=ADOBE_DEFLATE
  Level 7: tiled, tile=256x256, size=17x17, compression=ADOBE_DEFLATE
<OME xmlns="http://www.openmicroscopy.org/Schemas/OME/2016-06" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.openmicroscopy.org/Schemas/OME/2016-06 http://www.openmicroscopy.org/Schemas/OME/2016-06/ome.xsd" UUID="urn:uuid:c4f0b180-3298-11f1-8645-00155daed71b" Creator="tifffile.py 2025.5.10">
  <Image ID="Image:0" Name="Image0">
    <Pixels ID

In [14]:
from tifffile import TiffFile

path = "/mnt/e/SLIDE-0272_segment_merge_whole_cell.tiff"
with TiffFile(path) as tf:
    for s_idx, s in enumerate(tf.series):
        print(f"Series {s_idx}: axes={s.axes}, shape={s.shape!r}")
        # Most WSIs use levels (pyramids) within a series
        levels = getattr(s, "levels", [s])
        for l_idx, lvl in enumerate(levels):
            page = lvl.pages[0]
            tw = getattr(page, "tilewidth", None)
            tl = getattr(page, "tilelength", None)
            rps = getattr(page, "rowsperstrip", None)
            comp = page.compression.name if page.compression else "none"
            kind = "tiled" if (tw and tl) else "striped"
            ts = f"{tw}x{tl}" if (tw and tl) else f"rows/strip={rps}"
            print(f"  Level {l_idx}: {kind}, tile={ts}, size={lvl.shape[-2]}x{lvl.shape[-1]}, compression={comp}")


Series 0: axes=YX, shape=(60899, 59212)
  Level 0: tiled, tile=256x256, size=60899x59212, compression=ADOBE_DEFLATE
{"shape": [60899, 59212]}
